In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

#more imports
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
from tqdm import tqdm


# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28,28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=2)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(1, 5, figsize=(15, 5))


for i in range(5):
    idx = np.random.randint(0, len(train_dataset))
    img, label = train_dataset[idx]

    img_np = img.numpy().transpose(1, 2, 0)

    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {letters[label - 1]}')
    axes[i].axis('off')

plt.show()



In [ ]:
# Write your code here
model = efficientnet_v2_s(pretrained = True)

#modifying the classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features,26)

#moving to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch in tqdm(loader):
        images, labels = batch['image'], batch['label']
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        logits = model(images) #Get logits
        loss = criterion(logits, labels) #CEL already have softmax

        loss.backward() #Backward pass

        optimizer.step() #Update weights

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)

def evaluate(model, loader, criterion):
    model.eval() #Not updating weights
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for batch in tqdm(loader):
            images, labels = batch['image'], batch['label']
            images, labels = images.to(device), labels.to(device)

            logits = model(images) #Predict

            loss = criterion(logits, labels) #Calculate loss

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
#device already set

criterion = nn.CrossEntropyLoss() #Defining loss function, multiclass classification
learning_rate = 0.001
optimizer = optim.AdamW(model.parameters(), lr= learning_rate)
epochs = 10 #Number of epochs


#For visualization
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

# Training
for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    print(f"Epoch {epoch+1}/{epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={test_loss:.4f}, Val Accuracy={test_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, epochs+1), test_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, epochs+1), test_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
